<a href="https://colab.research.google.com/github/ancestor9/mathematics-for-machine-learning/blob/main/scripts/Topic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import random
import warnings
from sklearn.datasets import fetch_20newsgroups
!pip install gensim
import gensim
from gensim import corpora
from gensim.models import LdaModel
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer

# pyLDAvis 시각화 라이브러리
!pip install pyLDAvis
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# 경고 메시지 숨기기
warnings.filterwarnings('ignore', category=DeprecationWarning)
nltk.download('stopwords', quiet=True)

# =====================================================================
# 1. 데이터 로드 및 무작위 5개 주제 추출
# =====================================================================
all_categories = [
    'alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc',
    'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x',
    'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball',
    'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med',
    'sci.space', 'soc.religion.christian', 'talk.politics.guns',
    'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc'
]

# 무작위로 5개 주제 선정
random.seed(42)  # 재현성을 원하시면 주석을 해제하거나 변경하세요
selected_categories = random.sample(all_categories, 5)

print("=" * 60)
print(f"★ 정답 (무작위 선택된 5개 주제): {selected_categories}")
print("=" * 60)
print("데이터를 불러오는 중입니다...")

# 정답 데이터 로드 (시각화 분석 후 맞추기 위해 텍스트만 활용)
newsgroups = fetch_20newsgroups(subset='train', categories=selected_categories,
                                remove=('headers', 'footers', 'quotes'))
documents = newsgroups.data

# =====================================================================
# 2. 텍스트 전처리
# =====================================================================
print("텍스트 전처리를 진행합니다...")
tokenizer = RegexpTokenizer(r'\w+')
en_stopwords = set(stopwords.words('english'))

# 추가적인 무의미한 단어 필터링
add_stopwords = {'ax', 'max', 'g9v', 'b8f', 'a86', 'pl', '145', '1d9', '0t', '34u'}
en_stopwords = en_stopwords.union(add_stopwords)

processed_docs = []
for doc in documents:
    # 소문자 변환 및 토큰화
    tokens = tokenizer.tokenize(doc.lower())
    # 불용어 제거 및 3글자 이상 단어만 유지 (숫자 제외)
    stopped_tokens = [t for t in tokens if t not in en_stopwords and not t.isdigit() and len(t) > 2]
    processed_docs.append(stopped_tokens)

# Dictionary 및 Corpus 생성
dictionary = corpora.Dictionary(processed_docs)
# 지나치게 빈도가 낮거나 높은 단어 필터링 (품질 향상)
dictionary.filter_extremes(no_below=3, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in processed_docs]

# =====================================================================
# 3. 토픽 개수별(3개 ~ 7개) LDA 모델 학습 및 시각화 저장
# =====================================================================
for num_topics in range(3, 8):
    print(f"\n▶ 토픽 개수 [{num_topics}개] 모델 학습 및 시각화 생성 중...")

    # LDA 모델 학습
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=100,
        update_every=1,
        chunksize=100,
        passes=10,
        alpha='auto',
        per_word_topics=True
    )

    # pyLDAvis 준비
    vis_data = gensimvis.prepare(lda_model, corpus, dictionary, sort_topics=False)

    # HTML 파일로 저장
    file_name = f'lda_vis_{num_topics}_topics.html'
    pyLDAvis.save_html(vis_data, file_name)
    print(f"   ㄴ 완료! '{file_name}' 파일로 저장되었습니다.")

print("\n" + "=" * 60)
print("실습 준비 완료! 작업 디렉토리에 생성된 HTML 파일들을 브라우저로 열어보세요.")
print("어떤 토픽 개수(3~7개)가 원래 5개 주제의 의미를 가장 잘 분리하는지 찾아보세요!")
print("=" * 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 9.8 MB/s eta 0:00:00
★ 정답 (무작위 선택된 5개 주제): ['comp.sys.ibm.pc.hardware', 'alt.atheism', 'rec.motorcycles', 'rec.autos', 'talk.politics.guns']
데이터를 불러오는 중입니다...
텍스트 전처리를 진행합니다...

▶ 토픽 개수 [3개] 모델 학습 및 시각화 생성 중...
   ㄴ 완료! 'lda_vis_3_topics.html' 파일로 저장되었습니다.

▶ 토픽 개수 [4개] 모델 학습 및 시각화 생성 중...
   ㄴ 완료! 'lda_vis_4_topics.html' 파일로 저장되었습니다.

▶ 토픽 개수 [5개] 모델 학습 및 시각화 생성 중...
   ㄴ 완료! 'lda_vis_5_topics.html' 파일로 저장되었습니다.

▶ 토픽 개수 [6개] 모델 학습 및 시각화 생성 중...
   ㄴ 완료! 'lda_vis_6_topics.html' 파일로 저장되었습니다.

▶ 토픽 개수 [7개] 모델 학습 및 시각화 생성 중...
   ㄴ 완료! 'lda_vis_7_topics.html' 파일로 저장되었습니다.

실습 준비 완료! 작업 디렉토리에 생성된 HTML 파일들을 브라우저로 열어보세요.
어떤 토픽 개수(3~7개)가 원래 5개 주제의 의미를 가장 잘 분리하는지 찾아보세요!


### **Tokenization**

In [4]:
dictionary

In [6]:
for key, value in dictionary.items():
    print(f"{key}: {value}")

0: 16k
1: though
2: account
3: act
4: agree
5: bad
6: basically
7: benefits
8: best
9: cannot
10: choose
11: course
12: dead
13: death
14: etc
15: even
16: experienced
17: exposed
18: find
19: folk
20: general
21: given
22: hard
23: helped
24: individual
25: judge
26: least
27: might
28: people
29: personally
30: population
31: really
32: religion
33: religious
34: right
35: route
36: said
37: say
38: seemed
39: state
40: totally
41: try
42: use
43: useful
44: way
45: ways
46: well
47: whether
48: without
49: would
50: air
51: costs
52: fix
53: friend
54: gas
55: internal
56: know
57: leak
58: likes
59: needs
60: new
61: possible
62: problem
63: replacing
64: tercel
65: toyota
66: using
67: whole
68: address
69: advance
70: anyone
71: club
72: current
73: guzzi
74: mail
75: moto
76: national
77: owners
78: please
79: thanks
80: com
81: ever
82: fail
83: handheld
84: hope
85: jmd
86: spell
87: take
88: usa
89: company
90: fault
91: hell
92: hits
93: immediately
94: insurance
95: jerk
96

In [7]:
import pyLDAvis.gensim_models

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(ldamodel, corpus, dictionary)
pyLDAvis.display(vis)


NameError: name 'ldamodel' is not defined